In [1]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.settings import Settings
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.vector_stores.azureaisearch import AzureAISearchVectorStore
from llama_index.vector_stores.azureaisearch import IndexManagement, MetadataIndexFieldType

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

aoai_api_key = os.getenv("OPENAI_API_KEY")
print(f"Key loaded: {aoai_api_key[:5]}...")

llm = AzureOpenAI(
    model="gpt-4o",
    deployment_name="gpt-4o",
    api_key=aoai_api_key,
    azure_endpoint="https://bobbyragdemo.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-01-01-preview",
    api_version="2025-01-01-preview",
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-large",
    deployment_name="text-embedding-3-large",
    api_key=aoai_api_key,
    azure_endpoint="https://bobbyragdemo.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15",
    api_version="2023-05-15",
    dimensions=1536,
)

search_service_api_key = os.getenv("SEARCH_SERVICE_API_KEY")
print(f"Search service key loaded: {search_service_api_key[:5]}...")
search_service_endpoint = "https://bobby-rag-search.search.windows.net"
index_name = "bobbyragdemo-index"

index_client = SearchIndexClient(
    endpoint=search_service_endpoint,
    credential=AzureKeyCredential(search_service_api_key),
)

search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_service_api_key),
)

vector_store = AzureAISearchVectorStore(
    search_or_index_client=index_client,
    index_name=index_name,
    index_management=IndexManagement.CREATE_IF_NOT_EXISTS,
    id_field_key="id",
    chunk_field_key="chunk",
    embedding_field_key="embedding",
    metadata_string_field_key="metadata",
    doc_id_field_key="doc_id",
    embedding_dimensionality=1536,
)

async_search_or_index_client is None. Depending on the client type passed in, sync or async functions may not work.


Key loaded: 5RcQA...
Search service key loaded: K5V2l...


# Ingest Documents

In [3]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader("my_docs")
documents = reader.load_data()

Settings.llm = llm
Settings.embed_model = embed_model
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context)

2026-05-12 19:20:01,334 - INFO - HTTP Request: POST https://bobbyragdemo.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
2026-05-12 19:20:01,404 - INFO - Uploading remaining batch of size 6, current progress 6 of 6, accumulated size 0.20 MB
2026-05-12 19:20:01,407 - INFO - Request URL: 'https://bobby-rag-search.search.windows.net/indexes('bobbyragdemo-index')/docs/search.index?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '205865'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '7b6306e6-4e61-11f1-9bb9-224fa7c2e1a7'
    'User-Agent': 'azsdk-python-search-documents/11.5.1 Python/3.13.3 (macOS-15.7.5-arm64-arm-64bit-Mach-O) llamaindex-python'
A body is sent with the request
2026-05-12 19:20:01,738 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'


# Query Documents

In [4]:
query_engine = index.as_query_engine()
response = query_engine.query("What are some hard drive issues the author has experienced?")
print(response)

2026-05-12 19:20:10,167 - INFO - HTTP Request: POST https://bobbyragdemo.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
2026-05-12 19:20:10,172 - INFO - Vector search with supplied embedding
2026-05-12 19:20:10,185 - INFO - Request URL: 'https://bobby-rag-search.search.windows.net/indexes('bobbyragdemo-index')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '31168'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '809e7262-4e61-11f1-9bb9-224fa7c2e1a7'
    'User-Agent': 'azsdk-python-search-documents/11.5.1 Python/3.13.3 (macOS-15.7.5-arm64-arm-64bit-Mach-O) llamaindex-python'
A body is sent with the request
2026-05-12 19:20:10,251 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata

The author experienced the failure of a 2TB external hard drive. The log file indicates issues with the hard drive's file system, including "Keys out of order," "Invalid sibling link," and an unsuccessful attempt to rebuild the catalog B-tree. The volume named "apple archive" could not be repaired.


In [5]:
query_engine = index.as_query_engine()
response = query_engine.query("What are some skills the author has for IT management?")
print(response)

2026-05-12 19:20:21,327 - INFO - HTTP Request: POST https://bobbyragdemo.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
2026-05-12 19:20:21,327 - INFO - Vector search with supplied embedding
2026-05-12 19:20:21,335 - INFO - Request URL: 'https://bobby-rag-search.search.windows.net/indexes('bobbyragdemo-index')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '31191'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': '8743b870-4e61-11f1-9bb9-224fa7c2e1a7'
    'User-Agent': 'azsdk-python-search-documents/11.5.1 Python/3.13.3 (macOS-15.7.5-arm64-arm-64bit-Mach-O) llamaindex-python'
A body is sent with the request
2026-05-12 19:20:21,404 - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata

The author demonstrates several IT management skills, including:

1. **Database and File System Backup**: Proficiency in using shell scripts to back up databases and file systems.
2. **WordPress Administration**: Ability to update WordPress core and plugins using wp-cli.
3. **Web Analytics Management**: Knowledge of updating and configuring analytics tools, including renaming paths in zip files and deploying updates.
4. **File Compression and Optimization**: Capability to check and update gzip files using scripts.
5. **Server Setup and Configuration**: Experience in setting up an Ubuntu server with LAMP, OpenSSH, and configuring permissions for web directories.
6. **Apache Configuration**: Skills in enabling and configuring Apache modules like `include` and `rewrite`, and managing `.htaccess` files.
7. **Version Control**: Familiarity with Git, Subversion, and setting up SSH keys for GitHub integration.
8. **WordPress Installation and Plugin Management**: Expertise in installing WordPr